# Sentiment Classification Using Embeddings

This notebook performs sentiment classification on tweets using embedding-based features and machine learning models.

## Objectives
- Text preprocessing
- Exploratory Data Analysis (EDA)
- Generate embeddings using SentenceTransformers
- Train Logistic Regression and XGBoost
- Evaluate model performance
- Make custom predictions


In [ ]:
!pip install sentence-transformers xgboost seaborn wordcloud kagglehub

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
import kagglehub

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from xgboost import XGBClassifier
from wordcloud import WordCloud

print('Libraries loaded successfully')

In [ ]:
# Download dataset
path = kagglehub.dataset_download('yasserh/twitter-tweets-sentiment-dataset')
print('Dataset path:', path)
files = os.listdir(path)
print('Files:', files)

csv_file = [f for f in files if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))
df.head()

In [ ]:
# Clean dataset
df = df.dropna(subset=['text']).reset_index(drop=True)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['clean_text'] = df['text'].apply(clean_text)
df.head()

In [ ]:
# EDA
sns.countplot(x='sentiment', data=df)
plt.title('Sentiment Distribution')
plt.show()

df['length'] = df['clean_text'].apply(len)
sns.histplot(df['length'], bins=50)
plt.title('Text Length Distribution')
plt.show()

In [ ]:
# Wordcloud
text = ' '.join(df[df['sentiment']=='positive']['clean_text'])
wc = WordCloud(width=800, height=400).generate(text)
plt.imshow(wc)
plt.axis('off')
plt.show()

In [ ]:
# Generate embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df['clean_text'].tolist(), show_progress_bar=True)
print(embeddings.shape)

In [ ]:
# Labels
label_map = {'negative':0, 'neutral':1, 'positive':2}
df['label'] = df['sentiment'].map(label_map)

X_train, X_test, y_train, y_test = train_test_split(
    embeddings, df['label'], test_size=0.2, random_state=42
)

In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)
print('LR Accuracy:', accuracy_score(y_test, pred_lr))

In [ ]:
# XGBoost
xgb = XGBClassifier()
xgb.fit(X_train, y_train)
pred_xgb = xgb.predict(X_test)
print('XGB Accuracy:', accuracy_score(y_test, pred_xgb))

In [ ]:
# Evaluation
print(classification_report(y_test, pred_xgb))

cm = confusion_matrix(y_test, pred_xgb)
sns.heatmap(cm, annot=True, fmt='d')
plt.show()

## Insights
- Embeddings capture semantic meaning effectively.
- XGBoost performs better than Logistic Regression.
- Useful for brand monitoring and opinion analysis.
